## Single project

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame KITTI.
gt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/kitti_gt_annos_2/gt_lidar_to_camera_labels_2'

# Detection data transformed from BEV to Camera frame KITTI.
# check if code is correct, gt vs. gt (100% results)
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/kitti_gt_annos_2/gt_lidar_to_camera_labels_2'

# check Pytorch FP32 pred [-pi/4...3pi/4] vs. gt
dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_pt_fp32'
# check TRT FP32 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp32'
# check TRT FP16 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp16'
# check TRT INT8 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_int8'

# check pred [0...pi/2] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_regularized/pred_lidar_to_camera_fp32_rgd'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path)) # here gtlabels

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), # here detection labels
    current_class=[0, 1, 2], score_thresh=0.2,
    eval_roi=False) 

metrics = results['entire_area']

# BBox
# NOTE: Dependent on 3D calculation
# print("\n=== 2D BBox Metrics (IoU=0.7) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls}: Easy: {metrics[f'{cls}_bbox_easy_07']:.2f}, " + f"Moderate: {metrics[f'{cls}_bbox_mod_07']:.2f}, " + f"Hard: {metrics[f'{cls}_bbox_hard_07']:.2f}")
# print("\n=== 2D BBox Metrics (IoU=0.5) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls}: Easy: {metrics[f'{cls}_bbox_easy_05']:.2f}, " + f"Moderate: {metrics[f'{cls}_bbox_mod_05']:.2f}, " + f"Hard: {metrics[f'{cls}_bbox_hard_05']:.2f}")

# 3D 
# NOTE: Too inaccurate due to missing detector parameters
# print("\n=== 3D Metrics (IoU=0.7) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls} 3d AP: {metrics[f'{cls}_3d_easy_07']:.2f}, " + f"{metrics[f'{cls}_3d_mod_07']:.2f}, " + f"{metrics[f'{cls}_3d_hard_07']:.2f}\n" +
#     f"mAP = {(metrics[f'{cls}_3d_easy_07'] + metrics[f'{cls}_3d_mod_07'] + metrics[f'{cls}_3d_hard_07'])/3:.2f}")
# print("\n=== 3D Metrics (IoU=0.5) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls} 3d AP: {metrics[f'{cls}_3d_easy_05']:.2f}, " + f"{metrics[f'{cls}_3d_mod_05']:.2f}, " + f"{metrics[f'{cls}_3d_hard_05']:.2f}\n" +
#     f"mAP = {(metrics[f'{cls}_3d_easy_05'] + metrics[f'{cls}_3d_mod_05'] + metrics[f'{cls}_3d_hard_05'])/3:.2f}")

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")

# visualization source for Easy/Mod./Hard
# "Voting for Voting in Online Point Cloud Object Detection"

## Master thesis KITTI
### Testing KITTI default, densified, interpolated, upsampled predictions vs. GT

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame (KITTI format)
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/gts'

# check if code is correct, gt vs. gt (100% results), ATTENTION set score_thresh=-1
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/gts'

# Detection data transformed from BEV to Camera frame (KITTI format)
# Pytorch FP32 predictions
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds'
# Pytorch FP32 densified predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_densified'
# Pytorch FP32 interpolated predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_interpolated'
# Pytorch FP32 upsampled predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_upsampled'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=-1, # set this to -1, if you test gt vs. gt. Also KITTI default
    eval_roi=False) 

metrics = results['entire_area']

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")

# Future TODO: check if alpha values from predictions can be evaluated for AOS

## Master thesis ZOD
### Testing Zenseact default predictions vs. GT

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame (KITTI format)
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# check if code is correct, gt vs. gt (100% results), ATTENTION set score_thresh=-1
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# Detection data transformed from BEV to Camera frame (KITTI format)
# Pytorch FP32 predictions
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/default'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=-1, # set this to -1, if you test gt vs. gt. Also KITTI default
    eval_roi=False) 

metrics = results['entire_area']

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")

# NOTE: ZOD evaluation on KITTI, source: https://github.com/zenseact/EdgeAnnotationZChallenge/tree/main/eval
"""
vehicle_detection_BEV_AP : 72.898407
pedestrian_detection_BEV_AP : 48.200329
cyclist_detection_BEV_AP : 40.517864
vehicle_detection_3D_AP : 61.186440
pedestrian_detection_3D_AP : 41.484516
cyclist_detection_3D_AP : 37.284695
"""
# A reason for the lower AP values could be the difference on how truncation, occlusion and height is treated between the two datasets.

## Master thesis DG
### Testing KITTI trained models with DG methods to ZOD predictions vs. GT

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame (KITTI format)
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# check if code is correct, gt vs. gt (100% results), ATTENTION set score_thresh=-1
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# Detection data transformed from BEV to Camera frame (KITTI format)
# Pytorch FP32 default predictions
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/default'
# Pytorch FP32 densified predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/densified'
# Pytorch FP32 interpolated predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/interpolated'
# Pytorch FP32 upsampled predictions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/upsampled'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=-1, # set this to -1, if you test gt vs. gt. Also KITTI default
    eval_roi=False) 

metrics = results['entire_area']

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")



## DG = (AP_T - AP_S->T) / AP_T * 100

In [ ]:
from vod.evaluation import Evaluation
import os

def calculate_dg(ap_t, ap_s_to_t):
    return ((ap_t - ap_s_to_t) / ap_t * 100) if ap_t > 0 else 0

# GT data ZOD
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'
# AP_Target: ZOD model to ZOD data
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/default'

evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))
results_target = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=-1,
    eval_roi=False) 

metrics_target = results_target['entire_area']

# AP_S->T: KITTI default model to ZOD data
dt_path_source_to_target = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/default'
# AP_S->T: KITTI densified model to ZOD data
#dt_path_source_to_target = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/densified'
# AP_S->T: KITTI interpolated model to ZOD data
#dt_path_source_to_target = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/interpolated'
# AP_S->T: KITTI upsampled model to ZOD data
#dt_path_source_to_target = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/upsampled'

results_s2t = evaluation.evaluate(
    result_path=os.path.join(dt_path_source_to_target), 
    current_class=[0, 1, 2], 
    score_thresh=-1,
    eval_roi=False)

metrics_s2t = results_s2t['entire_area']

classes = ['Car', 'Pedestrian', 'Cyclist']
difficulties = ['easy', 'mod', 'hard']
iou_thresholds = ['05', '07']

print("\n" + "=" * 100)
print("DOMAIN GAP (DG) ANALYSIS: KITTI → ZOD")
print("=" * 100)
print("DG = (AP_T - AP_S→T) / AP_T × 100")
print("AP_T   = ZOD model on ZOD data")
print("AP_S→T = KITTI model on ZOD data")
print("=" * 100)

# Pro IoU Threshold
for iou in iou_thresholds:
    print(f"\n{'─' * 100}")
    print(f"IoU Threshold = {iou}")
    print(f"{'─' * 100}")
    
    # Table Header
    print(f"\n{'Class':<12} {'Difficulty':<12} {'AP_T':>10} {'AP_S→T':>10} {'DG (%)':>10}")
    print("─" * 56)
    
    for cls in classes:
        for diff in difficulties:
            key = f'{cls}_bev_{diff}_{iou}'
            
            ap_t = metrics_target[key]
            ap_s2t = metrics_s2t[key]
            dg = calculate_dg(ap_t, ap_s2t)
            
            print(f"{cls:<12} {diff.capitalize():<12} {ap_t:>10.2f} {ap_s2t:>10.2f} {dg:>10.2f}")
        
        avg_ap_t = sum(metrics_target[f'{cls}_bev_{d}_{iou}'] for d in difficulties) / 3
        avg_ap_s2t = sum(metrics_s2t[f'{cls}_bev_{d}_{iou}'] for d in difficulties) / 3
        avg_dg = calculate_dg(avg_ap_t, avg_ap_s2t)
        
        print(f"{cls:<12} {'Average':<12} {avg_ap_t:>10.2f} {avg_ap_s2t:>10.2f} {avg_dg:>10.2f}")
        print()

print("=" * 100)
print("OVERALL DOMAIN GAP (across all classes)")
print("=" * 100)

for iou in iou_thresholds:
    all_dg_values = []
    for cls in classes:
        for diff in difficulties:
            key = f'{cls}_bev_{diff}_{iou}'
            ap_t = metrics_target[key]
            ap_st = metrics_s2t[key]
            all_dg_values.append(calculate_dg(ap_t, ap_st))
    
    overall_dg = sum(all_dg_values) / len(all_dg_values)
    print(f"IoU={iou}: Overall Domain Gap = {overall_dg:.2f}%")

print("=" * 100)

## DGC_A = (AP(A)_S->T - AP_S->T) / (AP_T - AP_S->T) * 100
### A: Method

In [2]:
from vod.evaluation import Evaluation
import os

def calculate_dgc(ap_method, ap_baseline, ap_target):
    numerator = ap_method - ap_baseline
    denominator = ap_target - ap_baseline

    if denominator == 0:
        return 0.0
    
    return (numerator / denominator) * 100

# GT data ZOD
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# 1. AP_T: ZOD model on ZOD data (Target)
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/default'

evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))
results_target = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=-1,
    eval_roi=False) 

metrics_target = results_target['entire_area']

# 2. AP_S→T: KITTI default (baseline) on ZOD data
dt_path_baseline = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/default'
results_baseline = evaluation.evaluate(
    result_path=dt_path_baseline,
    current_class=[0, 1, 2],
    score_thresh=-1,
    eval_roi=False
)
metrics_baseline = results_baseline['entire_area']

methods = {
    'Densified': '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/densified',
    'Interpolated': '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/interpolated',
    'Upsampled': '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds/dg_methods/upsampled'
}

metrics_methods = {}
for method_name, dt_path in methods.items():
    results = evaluation.evaluate(
        result_path=dt_path,
        current_class=[0, 1, 2],
        score_thresh=-1,
        eval_roi=False
    )
    metrics_methods[method_name] = results['entire_area']

classes = ['Car', 'Pedestrian', 'Cyclist']
difficulties = ['easy', 'mod', 'hard']
iou_thresholds = ['05', '07']

for iou in iou_thresholds:
    print(f"\n{'─' * 120}")
    print(f"IoU Threshold = {iou}")
    print(f"{'─' * 120}")
    
    # Table header
    header = f"\n{'Class':<12} {'Difficulty':<12} {'AP_T':>8} {'AP_S→T':>8} "
    for method_name in methods.keys():
        header += f"{'AP^'+method_name[:3]:>8} {'DGC%':>8} "
    print(header)
    print("─" * 120)
    
    # Per class and difficulty
    for cls in classes:
        for diff in difficulties:
            key = f'{cls}_bev_{diff}_{iou}'
            
            ap_t = metrics_target[key]
            ap_baseline = metrics_baseline[key]
            
            row = f"{cls:<12} {diff.capitalize():<12} {ap_t:>8.2f} {ap_baseline:>8.2f} "
            
            for method_name in methods.keys():
                ap_method = metrics_methods[method_name][key]
                dgc = calculate_dgc(ap_method, ap_baseline, ap_t)
                row += f"{ap_method:>8.2f} {dgc:>7.1f}% "
            
            print(row)
        
        # Average per class
        avg_ap_t = sum(metrics_target[f'{cls}_bev_{d}_{iou}'] for d in difficulties) / 3
        avg_ap_baseline = sum(metrics_baseline[f'{cls}_bev_{d}_{iou}'] for d in difficulties) / 3
        
        row = f"{cls:<12} {'Average':<12} {avg_ap_t:>8.2f} {avg_ap_baseline:>8.2f} "
        
        for method_name in methods.keys():
            avg_ap_method = sum(metrics_methods[method_name][f'{cls}_bev_{d}_{iou}'] for d in difficulties) / 3
            avg_dgc = calculate_dgc(avg_ap_method, avg_ap_baseline, avg_ap_t)
            row += f"{avg_ap_method:>8.2f} {avg_dgc:>7.1f}% "
        
        print(row)
        print()

# Overall DGC
print("=" * 120)
print("OVERALL DOMAIN GAP CLOSURE (across all classes and difficulties)")
print("=" * 120)

for iou in iou_thresholds:
    print(f"\nIoU = {iou}:")
    
    for method_name in methods.keys():
        all_dgc = []
        
        for cls in classes:
            for diff in difficulties:
                key = f'{cls}_bev_{diff}_{iou}'
                ap_t = metrics_target[key]
                ap_baseline = metrics_baseline[key]
                ap_method = metrics_methods[method_name][key]
                
                dgc = calculate_dgc(ap_method, ap_baseline, ap_t)
                all_dgc.append(dgc)
        
        overall_dgc = sum(all_dgc) / len(all_dgc)
        print(f"  {method_name:12} DGC: {overall_dgc:6.1f}%")

print("=" * 120)

Evaluating kitti by default
mAP Image BBox finished
mAP BEV BBox finished
mAP 3D BBox finished
Evaluating kitti by default
mAP Image BBox finished
mAP BEV BBox finished
mAP 3D BBox finished
Evaluating kitti by default
mAP Image BBox finished
mAP BEV BBox finished
mAP 3D BBox finished
Evaluating kitti by default
mAP Image BBox finished
mAP BEV BBox finished
mAP 3D BBox finished
Evaluating kitti by default
mAP Image BBox finished
mAP BEV BBox finished
mAP 3D BBox finished

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
IoU Threshold = 05
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Class        Difficulty       AP_T   AP_S→T   AP^Den     DGC%   AP^Int     DGC%   AP^Ups     DGC% 
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Car          Easy            83.67    62.4